In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("--- Starting Step 2b: GeoLink Entity Resolution & Registry Matching ---")

# 1. Read input tables
df_unified = spark.read.table("inlap.silver.sites_unified")
df_geolink = spark.read.table("inlap.silver.geolink_conformed")

required_unified_columns = {
    "site_id", "street_address", "city", "state", "zip", "lat", "lon", "network_type", "status", "source_vendor"
}
required_geolink_columns = {
    "baseglid", "glid", "latitude", "longitude", "full_address", "unit_number", "city", "state", "zip",
    "location_type", "confidence_score", "last_verified"
}

missing_unified = sorted(required_unified_columns - set(df_unified.columns))
missing_geolink = sorted(required_geolink_columns - set(df_geolink.columns))

if missing_unified or missing_geolink:
    raise ValueError(
        f"Schema mismatch detected. Missing in sites_unified: {missing_unified}; "
        f"missing in geolink_conformed: {missing_geolink}"
    )

# 2. Standardize source-site records and registry records to the current project schema
df_source_records = (
    df_unified.select(
        F.col("site_id"),
        F.col("street_address"),
        F.col("city"),
        F.col("state"),
        F.col("zip"),
        F.col("lat").cast("double").alias("lat"),
        F.col("lon").cast("double").alias("lon"),
        F.col("network_type"),
        F.col("status"),
        F.col("source_vendor"),
    )
    .withColumn("lat_rounded", F.round(F.col("lat"), 4))
    .withColumn("lon_rounded", F.round(F.col("lon"), 4))
)

df_geolink_registry = df_geolink.select(
    F.coalesce(F.col("lat_rounded"), F.round(F.col("latitude"), 4)).alias("lat_rounded"),
    F.coalesce(F.col("lon_rounded"), F.round(F.col("longitude"), 4)).alias("lon_rounded"),
    F.col("baseglid"),
    F.col("glid"),
    F.col("full_address").alias("resolved_street_address"),
    F.col("unit_number").alias("resolved_unit_number"),
    F.col("city").alias("resolved_city"),
    F.col("state").alias("resolved_state"),
    F.col("zip").alias("resolved_zip"),
    F.col("location_type").alias("resolved_location_type"),
    F.col("confidence_score").alias("registry_confidence_score"),
    F.col("last_verified").alias("registry_last_verified"),
)

# 3. Build the coordinate-level registry rollup using the validated column names
coord_window = Window.partitionBy("lat_rounded", "lon_rounded").orderBy(
    F.when(F.col("glid") == F.col("baseglid"), F.lit(0)).otherwise(F.lit(1)).asc(),
    F.when(
        F.col("resolved_unit_number").isNull() | (F.trim(F.col("resolved_unit_number")) == ""),
        F.lit(0),
    ).otherwise(F.lit(1)).asc(),
    F.col("registry_confidence_score").desc(),
    F.col("registry_last_verified").desc(),
    F.col("glid").asc(),
)

df_geolink_representative = (
    df_geolink_registry
    .withColumn("coord_rank", F.row_number().over(coord_window))
    .filter(F.col("coord_rank") == 1)
    .drop("coord_rank")
    .withColumnRenamed("baseglid", "representative_baseglid")
    .withColumnRenamed("glid", "representative_glid")
)

df_geolink_coordinate_summary = (
    df_geolink_registry.groupBy("lat_rounded", "lon_rounded")
    .agg(
        F.countDistinct("baseglid").alias("matched_baseglid_count"),
        F.countDistinct("glid").alias("matched_glid_count"),
        F.max("registry_last_verified").alias("latest_registry_verification"),
        F.max("registry_confidence_score").alias("max_registry_confidence"),
    )
)

df_geolink_coordinate_rollup = df_geolink_coordinate_summary.join(
    df_geolink_representative,
    ["lat_rounded", "lon_rounded"],
    "left",
)

# 4. Match sites to the coordinate rollup using the same status rules as the upstream silver entity-resolution step
df_matched_registry = (
    df_source_records.join(df_geolink_coordinate_rollup, ["lat_rounded", "lon_rounded"], "left")
    .withColumn(
        "geolink_match_status",
        F.when(F.col("matched_glid_count").isNull(), F.lit("no_match"))
        .when(F.col("matched_glid_count") == 1, F.lit("single_glid"))
        .when(F.col("matched_baseglid_count") == 1, F.lit("ambiguous_multi_unit"))
        .otherwise(F.lit("ambiguous_multi_location")),
    )
    .withColumn(
        "resolved_baseglid",
        F.when(F.col("matched_baseglid_count") == 1, F.col("representative_baseglid")),
    )
    .withColumn(
        "resolved_glid",
        F.when(F.col("matched_glid_count") == 1, F.col("representative_glid"))
        .when(F.col("matched_baseglid_count") == 1, F.col("representative_baseglid")),
    )
    .withColumn(
        "registry_join_grain",
        F.when(F.col("geolink_match_status") == "single_glid", F.lit("coordinate_glid"))
        .when(F.col("geolink_match_status") == "ambiguous_multi_unit", F.lit("coordinate_baseglid"))
        .otherwise(F.lit("coordinate_only")),
    )
    .withColumn("canonical_street_address", F.coalesce(F.col("street_address"), F.col("resolved_street_address")))
    .withColumn("canonical_city", F.coalesce(F.col("city"), F.col("resolved_city")))
    .withColumn("canonical_state", F.coalesce(F.col("state"), F.col("resolved_state")))
    .withColumn("canonical_zip", F.coalesce(F.col("zip"), F.col("resolved_zip")))
)

fallback_master_id = F.sha2(
    F.concat_ws(
        "|",
        F.col("lat_rounded").cast("string"),
        F.col("lon_rounded").cast("string"),
        F.coalesce(F.col("canonical_state"), F.lit("UNK")),
        F.coalesce(F.col("canonical_zip"), F.lit("UNK")),
    ),
    256,
)

df_matched_registry = df_matched_registry.withColumn(
    "master_site_id",
    F.coalesce(F.col("site_id"), F.col("resolved_glid"), fallback_master_id),
)

for c in df_matched_registry.columns:
    df_matched_registry = df_matched_registry.withColumnRenamed(c, c.lower())

# 5. Materialize / Write output to silver.sites_matched_registry
df_matched_registry.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("inlap.silver.sites_matched_registry")

row_count = df_matched_registry.count()
print(f"Successfully created `inlap.silver.sites_matched_registry` with {row_count} rows.")
display(df_matched_registry.orderBy(F.col("source_vendor").asc(), F.col("site_id").asc()).limit(5))